## DATA CLEANSING NOTEBOOK

In [1]:
import os

In [2]:
curr_path = !pwd
curr_path = curr_path[0] + "/"
input_file = os.path.join(os.path.dirname(curr_path), '..', 'inputs', 'inventory_raw.csv')
input_file

'/Users/jonathanperalgort/Documents/network-inventory-cleaning-and-validation/notebooks/../inputs/inventory_raw.csv'

In [3]:
import re
import pandas as pd
import numpy as np
import ipaddress

from urllib.parse import urlparse

In [4]:
df_raw = pd.read_csv(input_file)
df_raw

,source_row_id,ip,hostname,fqdn,mac,owner,device_type,site,notes
0,1,192.168.010.005,HOST01,NaN,AA-BB-CC-DD-EE-FF,priya (platform) priya@corp.example.com,server,BLR Campus,db host
1,2,10.0.1.300,host-02,host-02.local,11-22-33-44-55-66,ops,NaN,HQ Bldg 1,edge gw?
2,3,10.0.1,host03,NaN,aabb.ccdd.eeff,jane@corp.example.com,switch,HQ-BUILDING-1,NaN
3,4,10.0.1.1.2,printer-01,NaN,00:11:22:33:44:55,Facilities,printer,HQ,NaN
4,5,fe80::1%eth0,iot-cam01,NaN,00:aa:bb:cc:dd:ee,sec,iot,Lab-1,camera PoE on port 3
5,6,127.0.0.1,local-test,NaN,NaN,NaN,NaN,NaN,NaN
6,7,169.254.10.20,host-apipa,NaN,NaN,NaN,NaN,NaN,NaN
7,8,10.10.10.10,srv-10,NaN,NaN,platform,server,BLR campus,NaN
8,9,abc.def.ghi.jkl,badhost,NaN,NaN,NaN,NaN,NaN,NaN
9,10,192.168.1.-1,neg,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
df_raw.columns

Index(['source_row_id', 'ip', 'hostname', 'fqdn', 'mac', 'owner',
       'device_type', 'site', 'notes'],
      dtype='object')

### IP

IP VERSION

In [6]:
# TODO: IMPLEMENT IP_VERSION CLASSIFICATION
def detect_ip_version(ip_str):
    if pd.isna(ip_str):
        return "invalid"
    s = str(ip_str).strip()

    if s == "":
        return "invalid"
    
    colon_count = s.count(':')
    dot_count = s.count('.')

    # Ipv6 indicators
    if colon_count >= 2:
        try:
            ipaddress.IPv6Address(s)
            return "6"
        except ipaddress.AddressValueError as e:
            return "invalid"
        
    elif dot_count == 3:
        try:
            ipaddress.IPv4Address(s)
            return "4"
        except ipaddress.AddressValueError as e:
            return "invalid"
        
    else:
        return "invalid"

VALIDATION & NORMALIZATION

In [7]:
def ipv4_validate_and_normalize(ip_str):
    if pd.isna(ip_str):
        return (False, None, "missing")
    s = str(ip_str).strip()
    if ':' in s:
        return (False, None, "ipv6_or_non_ipv4")
    parts = s.split(".")
    if len(parts) != 4:
        return (False, None, "wrong_octet_count")
    
    canonical_parts = []
    for p in parts:
        if p == '':
            return (False, None, "empty_octet")
        if not (p.lstrip("+").isdigit() and not p.startswith("-")):
            return (False, None, "non_numeric_or_negative")
        try:
            v = int(p, 10)
        except ValueError:
            return (False, None, "non_decimal_format")
        if v < 0 or v > 255:
            return (False, None, "octet_out_of_range")
        canonical_parts.append(str(v))
    canonical = ".".join(canonical_parts)
    
    return (True, canonical, "ok")

IP_TYPE

In [8]:
def classify_ipv4_type(ip):
    if pd.isna(ip) or ip is None:
        return "invalid"
    
    try:
        octets = list(map(int, ip.split('.')))

        # Checking private ranges
        if octets[0] == 10:
            return "private_rfc1918"
        elif octets[0] == 172 and 16 <= octets[1] <= 31:
            return "private_rfc1918"
        elif octets[0] == 192 and octets[1] == 168:
            return "private_rfc1918"
        
        # Other ranges
        elif octets[0] == 169 and octets[1] == 254:
            return "link_local_apipa"
        elif octets[0] == 127:
            return "loopback"
        
        return "public_or_other"
    except (ValueError, AttributeError, IndexError):
        return "invalid_format"

SUBNET_CIDR

In [9]:
def default_subnet(ip, ip_type=None):
    if pd.isna(ip) or ip is None:
        return ""
    
    try:
        octets = list(map(int, ip.split('.')))

        # Get Ip type, if not provided
        if ip_type is None:
            ip_type = classify_ipv4_type_pandas(ip)

        # Subnet strategies
        if ip_type == 'private_rfc1918':
            if octets[0] == 10:
                return f"10.0.0.0/8"
            elif octets[0] == 172:
                return f"172.{octets[1]}.0.0/16"
            elif octets[0] == 192 and octets[1] == 168:
                return f"192.168.{octets[2]}.0/24"
            
        elif ip_type == "link_local_apipa":
            return "169.254.0.0/16"

        elif ip_type == "loopback":
            return "127.0.0.0/8"
        
        elif ip_type == "public_or_other":
            # Assumign /24 for public IPs
            return f"{octets[0]}.{octets[1]}.{octets[2]}.0/24"
        
        return ""
    
    except (ValueError, AttributeError, IndexError):
        return ""

In [10]:
df_final = pd.DataFrame()
df_ipv4 = pd.DataFrame()
df_ipv4[['ip_valid', 'ip_canonical', 'ip_reason']] = df_raw['ip'].apply(
    lambda x: pd.Series(ipv4_validate_and_normalize(x))
)
df_ipv4['ip_type'] = df_ipv4['ip_canonical'].apply(
    lambda x: pd.Series(classify_ipv4_type(x))
)
df_ipv4['subnet_cidr'] = df_ipv4.apply(
    lambda x: default_subnet(x['ip_canonical'], x['ip_type'])
    if x['ip_valid'] else "",
    axis=1
)
df_final[['ip', 'ip_valid', 'ip_type', 'subnet_cidr']] = df_ipv4[['ip_canonical', 'ip_valid', 'ip_type', 'subnet_cidr']]
df_final['hostname'] = df_raw['hostname']
df_final

,ip,ip_valid,ip_type,subnet_cidr,hostname
0,192.168.10.5,True,private_rfc1918,192.168.10.0/24,HOST01
1,None,False,invalid,,host-02
2,None,False,invalid,,host03
3,None,False,invalid,,printer-01
4,None,False,invalid,,iot-cam01
5,127.0.0.1,True,loopback,127.0.0.0/8,local-test
6,169.254.10.20,True,link_local_apipa,169.254.0.0/16,host-apipa
7,10.10.10.10,True,private_rfc1918,10.0.0.0/8,srv-10
8,None,False,invalid,,badhost
9,None,False,invalid,,neg


### HOSTNAME

VALIDATION & NORMALIZATION

In [11]:
df_raw['hostname']

0         HOST01
1        host-02
2         host03
3     printer-01
4      iot-cam01
5     local-test
6     host-apipa
7         srv-10
8        badhost
9            neg
10         bcast
11         netid
12    dns-google
13       host-10
14    missing-ip
Name: hostname, dtype: object

In [12]:
def validate_hostname(hostname_str):
    """Validating hostname according to RFC1123"""
    if pd.isna(hostname_str):
        return (False, None, "missing")
    
    s = str(hostname_str).strip()

    if s == "":
        return (False, None, "empty_string")
    
    if len(s) > 63:
        return (False, s, "too_long")
    
    # Checking periods
    if '.' in s:
        return (False, s, "contains_periods")
    
    # Character validation - RFC 1123
    if not re.match(r'^[a-zA-Z0-9]([a-zA-Z0-9-]*[a-zA-Z0-9])?$', s):
        return (False, s, "invalid_characters")
    
    # Cannot start or end with hyphen
    if s.startswith('-') or s.endswith('-'):
        return (False, s, "hyphen_at_edge")
    
    # Normalize to lowercase for consistency
    canonical = s.lower()
    return (True, canonical, "ok")
    

In [13]:
df_hostname = pd.DataFrame()
df_hostname['hostname'] = df_raw['hostname']
df_hostname[['hostname_valid', 'hostname_canonical', 'hostname_reason']] = df_hostname['hostname'].apply(
    lambda x: pd.Series(validate_hostname(x))
)
df_hostname

,hostname,hostname_valid,hostname_canonical,hostname_reason
0,HOST01,True,host01,ok
1,host-02,True,host-02,ok
2,host03,True,host03,ok
3,printer-01,True,printer-01,ok
4,iot-cam01,True,iot-cam01,ok
5,local-test,True,local-test,ok
6,host-apipa,True,host-apipa,ok
7,srv-10,True,srv-10,ok
8,badhost,True,badhost,ok
9,neg,True,neg,ok


### FQDN

VALDIATION & NORMALIZATION

In [14]:
def validate_fqdn(fqdn_str, hostname_part=None, site_part=None):
    """
    Validates FQDN according to RFC 1035 and checks consistency with hostname/site.
    Returns a tuple: (is_valid: bool, normalized_value: str, error_code: str, consistency: str)
    """
    if pd.isna(fqdn_str) or fqdn_str is None:
        return (False, None, "missing", "inconsistent")
    
    s = str(fqdn_str).strip()

    if s == "":
        return (False, None, "empty_string", "inconsistent")
    
    # Remove trailing dot if present (optional in some systems)
    if s.endswith('.'):
        s = s[:-1]
    
    # Check overall length (RFC 1035: max 255 chars including dots)
    if len(s) > 255:
        return (False, s, "too_long", "inconsistent")
    
    # Split into labels
    labels = s.split('.')
    
    # Must have at least 2 labels (hostname + domain)
    if len(labels) < 2:
        return (False, s, "too_few_labels", "inconsistent")
    
    # Validate each label
    for i, label in enumerate(labels):
        # Check label length (max 63 chars per RFC 1035)
        if len(label) > 63:
            return (False, s, f"label_{i}_too_long", "inconsistent")
        
        # Check for empty labels (consecutive periods)
        if len(label) == 0:
            return (False, s, "empty_label", "inconsistent")
        
        # First and last label have special rules
        if i == 0:
            # First label is essentially the hostname - apply hostname rules
            if not re.match(r'^[a-zA-Z0-9]([a-zA-Z0-9-]*[a-zA-Z0-9])?$', label):
                return (False, s, "invalid_hostname_label", "inconsistent")
        else:
            # Other labels (domain parts) can start/end with digits
            # But still no leading/trailing hyphens
            if not re.match(r'^[a-zA-Z0-9]([a-zA-Z0-9-]*[a-zA-Z0-9])?$', label):
                # Allow digits at start/end for domain labels (like "2ndfloor" or "lab1")
                if not re.match(r'^[a-zA-Z0-9-]+$', label):
                    return (False, s, f"invalid_domain_label_{i}", "inconsistent")
        
        # Check for leading/trailing hyphens in all labels
        if label.startswith('-') or label.endswith('-'):
            return (False, s, f"label_{i}_hyphen_edge", "inconsistent")
    
    # Normalize to lowercase
    canonical = s.lower()
    
    # Check consistency with hostname and site if provided
    consistency_status = "unknown"
    if hostname_part is not None and site_part is not None:
        expected_fqdn = f"{hostname_part}.{site_part}".lower()
        if canonical == expected_fqdn:
            consistency_status = "consistent"
        else:
            consistency_status = "inconsistent"
    
    return (True, canonical, "valid", consistency_status)

In [27]:

def generate_reverse_ptr(ip):
    if pd.isna(ip) or ip is None:
        return None

    s_ip = str(ip).strip()
    if s_ip == "":
        return None

    try:
        ip_obj = ipaddress.ip_address(s_ip)
    except ValueError:
        return None

    reverse_key = ip_obj.reverse_pointer.lower().rstrip('.') 

    return reverse_key

In [29]:
df_fqdn = pd.DataFrame()
df_fqdn['fqdn'] = df_raw['fqdn']
df_fqdn['hostname_canonical'] = df_hostname['hostname_canonical']
df_fqdn['site'] = df_raw['site']
df_fqdn[['reverse_ptr']] = df_final['ip'].apply(
    lambda x: pd.Series(generate_reverse_ptr(x))
)
df_fqdn


,fqdn,hostname_canonical,site,reverse_ptr
0,NaN,host01,BLR Campus,5.10.168.192.in-addr.arpa
1,host-02.local,host-02,HQ Bldg 1,NaN
2,NaN,host03,HQ-BUILDING-1,NaN
3,NaN,printer-01,HQ,NaN
4,NaN,iot-cam01,Lab-1,NaN
5,NaN,local-test,NaN,1.0.0.127.in-addr.arpa
6,NaN,host-apipa,NaN,20.10.254.169.in-addr.arpa
7,NaN,srv-10,BLR campus,10.10.10.10.in-addr.arpa
8,NaN,badhost,NaN,NaN
9,NaN,neg,NaN,NaN


In [16]:
df_fqdn.iloc[0].fqdn, df_fqdn.iloc[0].hostname_canonical, df_fqdn.iloc[0].site

(nan, 'host01', 'BLR Campus')

In [17]:
fqdn_valid, fqdn_canonical, fqdn_error_reason, fqdn_consistency = validate_fqdn(df_fqdn.iloc[0].fqdn, df_fqdn.iloc[0].hostname_canonical, df_fqdn.iloc[0].site)
fqdn_valid, fqdn_canonical, fqdn_error_reason, fqdn_consistency

(False, None, 'missing', 'inconsistent')

In [18]:
df_fqdn[['fqdn_valid', 'fqdn_canonical', 'fqdn_error_reason', 'fqdn_consistency']] = (
    df_fqdn.apply(
        lambda x: validate_fqdn(x['fqdn'], x['hostname_canonical'], x['site']),
        axis='columns'
    ).apply(pd.Series)
)
df_fqdn

,fqdn,hostname_canonical,site,fqdn_valid,fqdn_canonical,fqdn_error_reason,fqdn_consistency
0,NaN,host01,BLR Campus,False,None,missing,inconsistent
1,host-02.local,host-02,HQ Bldg 1,True,host-02.local,valid,inconsistent
2,NaN,host03,HQ-BUILDING-1,False,None,missing,inconsistent
3,NaN,printer-01,HQ,False,None,missing,inconsistent
4,NaN,iot-cam01,Lab-1,False,None,missing,inconsistent
5,NaN,local-test,NaN,False,None,missing,inconsistent
6,NaN,host-apipa,NaN,False,None,missing,inconsistent
7,NaN,srv-10,BLR campus,False,None,missing,inconsistent
8,NaN,badhost,NaN,False,None,missing,inconsistent
9,NaN,neg,NaN,False,None,missing,inconsistent
